# Detecting inherent linearity in transformer models

This Notebook serves to experiment with detecting inherent linearity in Llama models. These experiments will be run on Llama-2-7B to test functionality and provide evidence for a feasibility study as part of the graduation preparation phase. Larger experiments for the final paper(s) will be handled using Python files in this repository.

In [ ]:
import torch
from transformers import LlamaForCausalLM, LlamaTokenizer
import matplotlib.pyplot as plt
import numpy as np
from huggingface_hub import login
from datasets import load_dataset
import re

# Log in to Hugging Face with hf.login file to access the model
login(token=open("../hf.login").read().strip())

# Load the Llama-2-7B model and tokenizer
model = LlamaForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf")
tokenizer = LlamaTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")
tokenizer.pad_token = tokenizer.eos_token # Set pad token to eos token if not already set to prevent errors

print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Model and tokenizer loaded.")

In [ ]:
# Load TinyStories dataset
from datasets import load_dataset
import re
import os

def clean_text(text):
    text = re.sub(r"[^a-zA-Z0-9\s.,!?']", "", text)  # Remove special characters
    text = re.sub(r"\s+", " ", text).strip()  # Remove extra spaces
    return text

# Tokenize and clean the dataset
def preprocess(examples):
    examples["text"] = [clean_text(t) for t in examples["text"]]
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)


def load_datasets():
    if not os.path.exists("./data"):
        os.makedirs("./data")

    if os.path.exists("./data/tiny_stories_train") and os.path.exists("./data/tiny_stories_val"):
        train_set = load_dataset("roneneldan/TinyStories", split="train").load_from_disk("./data/tiny_stories_train")
        val_set = load_dataset("roneneldan/TinyStories", split="validation").load_from_disk("./data/tiny_stories_val")
        print("Datasets loaded from disk.")
        return train_set, val_set

    train_set = load_dataset("roneneldan/TinyStories", split="train")
    val_set = load_dataset("roneneldan/TinyStories", split="validation")

    train_set = train_set.map(preprocess, batched=True)
    val_set = val_set.map(preprocess, batched=True)
    print("Datasets loaded and preprocessed.")

    # Save to disk for faster loading later
    train_set.save_to_disk("./data/tiny_stories_train")
    val_set.save_to_disk("./data/tiny_stories_val")
    print("Datasets saved to disk.")

    return train_set, val_set

train_dataset, val_dataset = load_datasets()

In [ ]:
# Do a forward pass to ensure everything is working
def forward_pass(model, tokenizer, dataset, device='cuda', debug=False):
    model.to(device)
    model.eval()
    if debug:
        print("Preparing inputs for forward pass...")
    inputs = tokenizer(dataset[0]['text'], return_tensors='pt', truncation=True, padding='max_length', max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    if debug:
        print("Inputs prepared. Performing forward pass...")
    with torch.no_grad():
        outputs = model(**inputs)
    if debug:
        print("Forward pass successful.")
    return outputs

outputs = forward_pass(model, tokenizer, val_dataset, debug=True)
print("Forward pass outputs:", outputs)